# Exception Handling

A reference notebook covering Python exception handling:

- `try` and `except`
- `else`
- `finally`
- `raise`
- Custom exceptions
- Multiple exceptions
- Exception objects and useful diagnostics
- Exception chaining
- Common patterns and edge cases

## 1. What Is an Exception?

An exception is an event raised during program execution when Python cannot continue normally with the current operation.

Examples include:

- `ValueError` — a value has an inappropriate form
- `TypeError` — an operation receives an inappropriate type
- `ZeroDivisionError` — division by zero
- `IndexError` — sequence index is out of range
- `KeyError` — dictionary key does not exist
- `FileNotFoundError` — requested file does not exist

If an exception is not handled, it propagates up the call stack and can terminate the program.

In [1]:
number = 10

# This raises ZeroDivisionError.
# result = number / 0

print("Program continues because the failing line is commented out.")

Program continues because the failing line is commented out.


## 2. Basic `try` and `except`

Put code that may fail inside a `try` block.

An `except` block specifies how to handle an exception.

In [2]:
try:
    value = int("100")
    print(value)
except ValueError:
    print("The value could not be converted to an integer.")

100


## 3. Handling an Actual Exception

The exception handler runs only when a matching exception occurs.

In [3]:
try:
    value = int("not a number")
    print(value)
except ValueError:
    print("Invalid integer input.")

Invalid integer input.


## 4. Catching the Exception Object

Use `as` to store the exception object in a variable.

The exception object often contains useful diagnostic information.

In [4]:
try:
    result = 10 / 0
except ZeroDivisionError as error:
    print("Exception type:", type(error).__name__)
    print("Exception message:", error)

Exception type: ZeroDivisionError
Exception message: division by zero


## 5. Multiple `except` Blocks

Different exception types can have different handling logic.

Python checks the handlers from top to bottom and uses the first matching handler.

In [5]:
value = "abc"

try:
    number = int(value)
    result = 100 / number
except ValueError:
    print("The input is not a valid integer.")
except ZeroDivisionError:
    print("Cannot divide by zero.")

The input is not a valid integer.


## 6. Handling Multiple Exception Types Together

A single `except` block can handle multiple exception types by using a tuple.

In [6]:
def divide_from_text(value):
    try:
        number = int(value)
        return 100 / number
    except (ValueError, ZeroDivisionError):
        return None

print(divide_from_text("10"))
print(divide_from_text("abc"))
print(divide_from_text("0"))

10.0
None
None


## 7. Exception Hierarchy

Exceptions form an inheritance hierarchy.

For example, many common runtime exceptions inherit from `Exception`.

Be careful when catching broad types: a handler for a parent exception can also catch its subclasses.

In [7]:
try:
    int("abc")
except ValueError:
    print("ValueError handler")
except Exception:
    print("General Exception handler")

ValueError handler


## 8. `except Exception` vs Bare `except`

`except Exception` catches most ordinary application exceptions.

A bare `except:` also catches exceptions such as `KeyboardInterrupt` and `SystemExit`, which is usually undesirable in application code.

Prefer catching specific exceptions when possible.

In [8]:
try:
    value = int("abc")
except Exception as error:
    print("Handled:", type(error).__name__)

# Generally avoid:
#
# try:
#     ...
# except:
#     ...

Handled: ValueError


## 9. The `else` Block

The `else` block runs only if the `try` block completes without raising an exception.

This is useful for code that should execute only after the risky operation succeeds.

In [9]:
try:
    number = int("42")
except ValueError:
    print("Invalid number.")
else:
    print("Conversion succeeded.")
    print("Number:", number)

Conversion succeeded.
Number: 42


## 10. Why Use `else`?

Keeping success-only code in `else` makes the boundary of the protected operation clearer.

If the code inside `else` raises an exception, it is not handled by the preceding `except` blocks.

In [10]:
try:
    number = int("42")
except ValueError:
    print("Input conversion failed.")
else:
    # This is outside the try block.
    result = 100 / number
    print(result)

2.380952380952381


## 11. The `finally` Block

`finally` runs whether the operation succeeds or an exception occurs.

It is commonly used for cleanup.

In [11]:
try:
    print("Performing an operation...")
except Exception:
    print("Something went wrong.")
finally:
    print("Cleanup runs here.")

Performing an operation...
Cleanup runs here.


## 12. `finally` with an Exception

`finally` still executes when an exception is raised.

If the exception is not otherwise handled, it continues propagating after `finally` completes.

In [12]:
try:
    print("Before error")
    # raise ValueError("Something went wrong")
except ValueError:
    print("Handled ValueError")
finally:
    print("Finally always runs")

Before error
Finally always runs


## 13. Full `try` / `except` / `else` / `finally`

The blocks have distinct roles:

```text
try      -> risky operation
except   -> handle failure
else     -> run after successful try
finally  -> cleanup regardless of outcome
```

In [13]:
try:
    value = int("25")
except ValueError as error:
    print("Conversion failed:", error)
else:
    print("Converted value:", value)
finally:
    print("Finished operation.")

Converted value: 25
Finished operation.


## 14. `raise`

Use `raise` to explicitly trigger an exception.

This is useful when input violates a function's requirements.

In [14]:
def set_age(age):
    if age < 0:
        raise ValueError("Age cannot be negative.")
    return age

print(set_age(21))

# set_age(-5)  # ValueError

21


## 15. Raising Different Exception Types

Choose an exception type that communicates the problem clearly.

In [15]:
def get_item(items, index):
    if not isinstance(index, int):
        raise TypeError("index must be an integer")

    if index < 0 or index >= len(items):
        raise IndexError("index is out of range")

    return items[index]

print(get_item(["a", "b", "c"], 1))

b


## 16. Re-Raising an Exception

Inside an `except` block, a bare `raise` re-raises the currently handled exception.

This is useful when logging or performing partial handling before allowing the caller to deal with the failure.

In [16]:
def process():
    try:
        int("abc")
    except ValueError:
        print("Logging conversion failure...")
        raise

try:
    process()
except ValueError:
    print("Caller handled the exception.")

Logging conversion failure...
Caller handled the exception.


## 17. Custom Exceptions

A custom exception is normally created by subclassing `Exception`.

Custom exceptions make domain-specific failures easier to identify and handle.

In [17]:
class InvalidAgeError(Exception):
    pass

def validate_age(age):
    if age < 0:
        raise InvalidAgeError("Age cannot be negative.")

    return age

try:
    validate_age(-1)
except InvalidAgeError as error:
    print("Invalid age:", error)

Invalid age: Age cannot be negative.


## 18. Custom Exception with Extra Data

Custom exception classes can store additional information.

In [18]:
class InsufficientBalanceError(Exception):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount

        super().__init__(
            f"Balance {balance} is insufficient for withdrawal {amount}."
        )

try:
    raise InsufficientBalanceError(100, 250)
except InsufficientBalanceError as error:
    print(error)
    print("Balance:", error.balance)
    print("Requested:", error.amount)

Balance 100 is insufficient for withdrawal 250.
Balance: 100
Requested: 250


## 19. Custom Exception Hierarchies

Custom exceptions can have their own inheritance hierarchy.

This allows callers to handle a broad application error or a more specific error.

In [19]:
class ApplicationError(Exception):
    pass

class ValidationError(ApplicationError):
    pass

class AuthenticationError(ApplicationError):
    pass

try:
    raise ValidationError("Invalid input.")
except ValidationError as error:
    print("Validation:", error)
except ApplicationError as error:
    print("Application error:", error)

Validation: Invalid input.


## 20. Exception Chaining with `from`

When one exception occurs while handling another, `raise ... from ...` explicitly records the original exception as the cause.

In [20]:
def parse_number(text):
    try:
        return int(text)
    except ValueError as error:
        raise ValueError("Could not parse the supplied number.") from error

try:
    parse_number("abc")
except ValueError as error:
    print(error)
    print("Original cause:", error.__cause__)

Could not parse the supplied number.
Original cause: invalid literal for int() with base 10: 'abc'


## 21. Suppressing Exception Context

`raise ... from None` can suppress the displayed exception context when the original implementation detail should not be exposed.

In [21]:
def parse_identifier(value):
    try:
        return int(value)
    except ValueError:
        raise ValueError("Identifier must be numeric.") from None

try:
    parse_identifier("abc")
except ValueError as error:
    print(error)

Identifier must be numeric.


## 22. `finally` and Resource Cleanup

`finally` is useful when a resource must be cleaned up regardless of success or failure.

For files and similar resources, context managers (`with`) are usually preferable because they automate cleanup.

In [22]:
file = None

try:
    file = open("example_temp.txt", "w", encoding="utf-8")
    file.write("Hello")
finally:
    if file is not None:
        file.close()

print("File closed.")

File closed.


## 23. Prefer Context Managers for Files

The `with` statement automatically handles cleanup when leaving the block, including when an exception occurs.

In [23]:
with open("example_temp.txt", "w", encoding="utf-8") as file:
    file.write("Hello from a context manager.")

print("The file was automatically closed.")

The file was automatically closed.


## 24. Exception Handling Around User Input

A common pattern is to validate conversion and retry when necessary.

In [24]:
def parse_score(text):
    try:
        score = float(text)
    except ValueError:
        return None

    if not 0 <= score <= 100:
        return None

    return score

print(parse_score("95.5"))
print(parse_score("abc"))
print(parse_score("120"))

95.5
None
None


## 25. Avoid Catching Exceptions Too Broadly

This pattern hides the actual problem:

```python
try:
    ...
except Exception:
    pass
```

Silently ignoring failures can make debugging difficult.

Catch specific exceptions and handle them intentionally.

In [25]:
def safe_integer(text):
    try:
        return int(text)
    except ValueError:
        return None

print(safe_integer("100"))
print(safe_integer("not a number"))

100
None


## 26. `pass` in an Exception Handler

`pass` can intentionally do nothing, but it should be used carefully.

If an exception is intentionally ignored, make sure that behavior is actually appropriate.

In [26]:
def optional_integer(text):
    try:
        return int(text)
    except ValueError:
        pass

    return None

print(optional_integer("42"))
print(optional_integer("unknown"))

42
None


## 27. Assertions vs Exceptions

`assert` is primarily intended for internal program assumptions and debugging.

For validating normal external input, explicitly raise an appropriate exception instead.

In [27]:
def set_percentage(value):
    if not 0 <= value <= 100:
        raise ValueError("Percentage must be between 0 and 100.")
    return value

print(set_percentage(75))

75


## 28. Exception Handling in Functions

A function can either handle an exception itself or allow it to propagate to its caller.

The appropriate choice depends on which layer has enough context to recover from the error.

In [28]:
def load_integer(text):
    return int(text)

try:
    number = load_integer("123")
    print(number)
except ValueError:
    print("The caller decided how to handle invalid input.")

123


## 29. Exception Propagation

If a function does not handle an exception, Python propagates it to the caller.

This continues up the call stack until a matching handler is found.

In [29]:
def level_three():
    raise ValueError("Failure in level three")

def level_two():
    level_three()

def level_one():
    level_two()

try:
    level_one()
except ValueError as error:
    print("Caught at the outer level:", error)

Caught at the outer level: Failure in level three


## 30. Practical Example: Safe Division

A reusable function can convert expected arithmetic failures into a predictable result.

In [30]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
None


## 31. Practical Example: Data Conversion

When processing data, conversion errors are often expected and should be handled explicitly.

In [31]:
raw_values = ["10", "20", "bad", "30"]

numbers = []

for value in raw_values:
    try:
        numbers.append(int(value))
    except ValueError:
        print(f"Skipping invalid value: {value}")

print(numbers)

Skipping invalid value: bad
[10, 20, 30]


## 32. Practical Example: Validation with a Custom Exception

Custom exceptions are useful when a validation failure has business or application meaning.

In [32]:
class InvalidScoreError(ValueError):
    pass

def validate_score(score):
    if not isinstance(score, (int, float)):
        raise TypeError("Score must be numeric.")

    if not 0 <= score <= 100:
        raise InvalidScoreError("Score must be between 0 and 100.")

    return score

for score in [95, -10, 110]:
    try:
        print("Valid:", validate_score(score))
    except InvalidScoreError as error:
        print("Invalid score:", error)

Valid: 95
Invalid score: Score must be between 0 and 100.
Invalid score: Score must be between 0 and 100.


## 33. Exception Information

Useful attributes and functions include:

- `type(error).__name__` — exception class name
- `str(error)` — human-readable message
- `repr(error)` — representation of the exception
- `error.__cause__` — explicit chained cause
- `error.__context__` — exception that occurred while handling another exception

In [33]:
try:
    int("abc")
except ValueError as error:
    print("Name:", type(error).__name__)
    print("str:", str(error))
    print("repr:", repr(error))
    print("Cause:", error.__cause__)
    print("Context:", error.__context__)

Name: ValueError
str: invalid literal for int() with base 10: 'abc'
repr: ValueError("invalid literal for int() with base 10: 'abc'")
Cause: None
Context: None


## 34. `try` Block Scope

Python does not create a separate scope for `try`, `except`, `else`, or `finally`.

Variables assigned in those blocks can remain available afterward if execution reached the assignment.

In [34]:
try:
    value = 42
except Exception:
    value = 0

print(value)

42


## 35. A Subtle `finally` Behavior

A `return` inside `finally` can override a return value or even suppress an exception.

For this reason, returning from `finally` is generally discouraged.

In [35]:
def example():
    try:
        return "from try"
    finally:
        # Returning here overrides the earlier return.
        return "from finally"

print(example())

from finally


## 36. `finally` Runs Before Propagation

Even when an exception is not caught by the local handlers, `finally` runs before the exception continues upward.

In [36]:
def operation():
    try:
        raise RuntimeError("Failure")
    finally:
        print("Cleanup before propagation")

try:
    operation()
except RuntimeError as error:
    print("Caught by caller:", error)

Cleanup before propagation
Caught by caller: Failure


## 37. Exception Handling and `else`

A useful structural pattern is:

```python
try:
    risky_operation()
except SpecificError:
    handle_failure()
else:
    handle_success()
finally:
    cleanup()
```

The `else` block is not a replacement for `try`; it specifically separates successful follow-up work from the risky operation.

In [37]:
def convert(text):
    try:
        number = int(text)
    except ValueError:
        print("Invalid input")
    else:
        print("Success:", number)
    finally:
        print("Conversion attempt finished")

convert("25")
convert("abc")

Success: 25
Conversion attempt finished
Invalid input
Conversion attempt finished


## 38. Common Exception Types

| Exception | Typical cause |
|---|---|
| `ValueError` | Correct type, inappropriate value |
| `TypeError` | Incompatible type |
| `ZeroDivisionError` | Division or modulo by zero |
| `IndexError` | Sequence index out of range |
| `KeyError` | Missing dictionary key |
| `AttributeError` | Missing attribute |
| `NameError` | Unknown variable name |
| `FileNotFoundError` | Missing file |
| `ImportError` | Import-related failure |
| `RuntimeError` | Generic runtime failure |

## 39. Common Pitfalls

1. Catching `Exception` when a specific exception is available.
2. Using bare `except:` unnecessarily.
3. Silently ignoring exceptions with `pass`.
4. Putting too much code inside `try`, making unrelated failures look like the same error.
5. Using `return` inside `finally`.
6. Using exceptions for ordinary control flow when a simple condition is clearer.
7. Hiding the original cause instead of using exception chaining where useful.
8. Creating custom exceptions that do not communicate a meaningful domain-level failure.

In [38]:
# Prefer a narrow try block.

text = "100"

try:
    number = int(text)
except ValueError:
    print("Only the conversion is protected.")
else:
    result = number * 2

print(result)

200


## 40. Quick Reference

### Basic structure

```python
try:
    risky_operation()
except SpecificError as error:
    handle_error(error)
else:
    handle_success()
finally:
    cleanup()
```

### Raise an exception

```python
raise ValueError("Invalid value")
```

### Custom exception

```python
class MyError(Exception):
    pass
```

### Re-raise

```python
except ValueError:
    raise
```

### Chain exceptions

```python
except ValueError as error:
    raise RuntimeError("Higher-level failure") from error
```